# Seleccion de Caracteristicas Multimetodo (Optimizado)
Objetivo: seleccionar variables para `tipo_violencia` y `nivel_riesgo_victima` con una estrategia multimetodo, controlando el consumo de RAM y exportando resultados en CSV.

## 1) Librerias

In [ ]:
# !pip install pandas pyarrow numpy scipy scikit-learn matplotlib seaborn
import gc
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import mutual_info_classif, RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 180)
sns.set(style="whitegrid")


## 2) Carga de datos

In [ ]:
PARQUET_PATH = "/home/munasqa/MAESTRIA/opencode/base_modelado.parquet"
df = pd.read_parquet(PARQUET_PATH)

if "nivel_de_riesgo_victima" in df.columns and "nivel_riesgo_victima" not in df.columns:
    df = df.rename(columns={"nivel_de_riesgo_victima": "nivel_riesgo_victima"})

targets = ["tipo_violencia", "nivel_riesgo_victima"]
for t in targets:
    if t not in df.columns:
        raise ValueError(f"Falta target: {t}")

print("Shape base:", df.shape)
display(df[targets].head())


## 3) Preparacion de features (ahorro de RAM)
- Se excluyen targets y columnas de leakage obvio.
- Se tipifican categoricas y numericas.
- Se aplica downcast numerico (`float32`/`int32`) para reducir memoria.

In [ ]:
leakage_obvio = ["tipo_violencia", "nivel_riesgo_victima", "tipo_violencia_lbl", "nivel_riesgo_victima_lbl"]
features = [c for c in df.columns if c not in leakage_obvio]
X_raw = df[features].copy()

for c in X_raw.columns:
    if str(X_raw[c].dtype) in ["object", "category"]:
        X_raw[c] = X_raw[c].astype(str).fillna("desconocido")
    else:
        X_raw[c] = pd.to_numeric(X_raw[c], errors="coerce")
        med = X_raw[c].median()
        X_raw[c] = X_raw[c].fillna(med if pd.notna(med) else 0)
        X_raw[c] = pd.to_numeric(X_raw[c], downcast="float")

print("Features candidatas:", len(features))
print("Memoria aprox X_raw (MB):", round(X_raw.memory_usage(deep=True).sum() / 1024**2, 2))


## 4) Que hace cada algoritmo (explicado simple)

- **Cramer's V (filtro)**: mide asociacion entre una feature categorica y el target. Cerca de 0 = poca asociacion; cerca de 1 = alta asociacion.
- **Mutual Information (filtro)**: mide cuanta informacion aporta una variable sobre el target, capturando relaciones no lineales. Mayor MI = mayor utilidad potencial.
- **Random Forest Importance (embedded)**: el bosque asigna importancia segun cuanto ayuda una variable a reducir impureza al hacer splits.
- **Permutation Importance (embedded/model-agnostic)**: con el modelo ya entrenado, se desordena una variable y se mide cuanto cae el desempeño. Si cae mucho, la variable era importante.
- **RFECV (wrapper)**: elimina variables en pasos, reentrena y valida con CV para encontrar el subconjunto que mejor rendimiento logra.

Finalmente, se normalizan rankings y se crea un **score de consenso** ponderado para tener un ranking robusto.

## 5) Funciones auxiliares

In [ ]:
def cramers_v(x, y):
    tab = pd.crosstab(x, y)
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return np.nan, np.nan
    chi2, p, _, _ = chi2_contingency(tab)
    n = tab.values.sum()
    r, k = tab.shape
    den = min(r - 1, k - 1)
    v = np.sqrt((chi2 / n) / den) if den > 0 else np.nan
    return v, p


def cap_cardinality(series, top_n=80):
    s = series.astype(str)
    keep = s.value_counts(dropna=False).head(top_n).index
    return s.where(s.isin(keep), "otros")


def to_ordinal_encoded(X_df):
    X = X_df.copy()
    cat_cols = [c for c in X.columns if str(X[c].dtype) in ["object", "category"]]
    num_cols = [c for c in X.columns if c not in cat_cols]

    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    if cat_cols:
        X[cat_cols] = enc.fit_transform(X[cat_cols]).astype(np.float32)

    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce").fillna(pd.to_numeric(X[c], errors="coerce").median())
        X[c] = pd.to_numeric(X[c], downcast="float")

    return X


def rank_to_score(df_rank, score_col, feature_col="feature", descending=True):
    r = df_rank[[feature_col, score_col]].dropna().copy()
    if r.empty:
        return pd.DataFrame(columns=["feature", "score"])
    r = r.sort_values(score_col, ascending=not descending).reset_index(drop=True)
    r["score"] = np.linspace(1.0, 0.0, len(r), endpoint=False)
    return r[[feature_col, "score"]]


def run_feature_selection_for_target(df_all, X_all, target, sample_size=50000, random_state=42):
    y_full = pd.to_numeric(df_all[target], errors="coerce")
    mask = y_full.notna()
    y_full = y_full[mask].astype(int)
    X_full = X_all.loc[mask]

    if len(y_full) > sample_size:
        _, X_use, _, y = train_test_split(
            X_full,
            y_full,
            test_size=sample_size,
            stratify=y_full,
            random_state=random_state,
        )
    else:
        X_use = X_full.copy()
        y = y_full.copy()

    print(f"Target: {target} | muestra usada: {len(y)}")

    rows_cv = []
    y_cat = y.astype(str)
    for c in X_use.columns:
        vals = cap_cardinality(X_use[c], top_n=80)
        v, p = cramers_v(vals, y_cat)
        rows_cv.append((c, v, p))
    r_cv = pd.DataFrame(rows_cv, columns=["feature", "cramers_v", "p_value"]).dropna()

    X_enc = to_ordinal_encoded(X_use)
    mi_vals = mutual_info_classif(X_enc, y, discrete_features="auto", random_state=random_state)
    r_mi = pd.DataFrame({"feature": X_enc.columns, "mi": mi_vals})

    Xtr, Xte, ytr, yte = train_test_split(X_enc, y, test_size=0.2, stratify=y, random_state=random_state)

    rf = RandomForestClassifier(
        n_estimators=120,
        max_depth=18,
        min_samples_leaf=5,
        random_state=random_state,
        n_jobs=2,
        class_weight="balanced_subsample",
    )
    rf.fit(Xtr, ytr)
    r_rf = pd.DataFrame({"feature": X_enc.columns, "rf_importance": rf.feature_importances_})

    top_perm = r_rf.sort_values("rf_importance", ascending=False).head(min(40, len(r_rf)))["feature"].tolist()
    rf_perm = RandomForestClassifier(
        n_estimators=80,
        max_depth=16,
        min_samples_leaf=5,
        random_state=random_state,
        n_jobs=2,
        class_weight="balanced_subsample",
    )
    rf_perm.fit(Xtr[top_perm], ytr)
    perm = permutation_importance(
        rf_perm,
        Xte[top_perm],
        yte,
        n_repeats=3,
        random_state=random_state,
        n_jobs=2,
        scoring="f1_macro",
    )
    r_perm = pd.DataFrame({"feature": top_perm, "perm_importance": perm.importances_mean})

    top_for_rfe = r_mi.sort_values("mi", ascending=False).head(min(25, len(r_mi)))["feature"].tolist()
    X_rfe = X_enc[top_for_rfe].copy()
    rfe_est = RandomForestClassifier(
        n_estimators=80,
        max_depth=14,
        min_samples_leaf=6,
        random_state=random_state,
        n_jobs=2,
        class_weight="balanced_subsample",
    )
    rfecv = RFECV(
        estimator=rfe_est,
        step=5,
        cv=3,
        scoring="f1_macro",
        min_features_to_select=5,
        n_jobs=2,
    )
    rfecv.fit(X_rfe, y)
    r_rfe = pd.DataFrame({"feature": X_rfe.columns, "rfe_rank": rfecv.ranking_, "rfe_selected": rfecv.support_.astype(int)})

    s_cv = rank_to_score(r_cv, "cramers_v", descending=True)
    s_mi = rank_to_score(r_mi, "mi", descending=True)
    s_rf = rank_to_score(r_rf, "rf_importance", descending=True)
    s_perm = rank_to_score(r_perm, "perm_importance", descending=True)
    s_rfe = r_rfe[["feature", "rfe_selected"]].copy()
    s_rfe["score"] = s_rfe["rfe_selected"].astype(float)
    s_rfe = s_rfe[["feature", "score"]]

    pool = pd.DataFrame({"feature": X_enc.columns}).drop_duplicates()
    for name, s in [("cv", s_cv), ("mi", s_mi), ("rf", s_rf), ("perm", s_perm), ("rfe", s_rfe)]:
        pool = pool.merge(s.rename(columns={"score": f"score_{name}"}), on="feature", how="left")

    pool = pool.fillna(0)
    pool["score_consenso"] = (
        0.20 * pool["score_cv"]
        + 0.25 * pool["score_mi"]
        + 0.25 * pool["score_rf"]
        + 0.20 * pool["score_perm"]
        + 0.10 * pool["score_rfe"]
    )
    pool = pool.sort_values("score_consenso", ascending=False).reset_index(drop=True)

    yhat = rf.predict(Xte)
    f1m = f1_score(yte, yhat, average="macro")

    out = {
        "cv": r_cv.sort_values("cramers_v", ascending=False),
        "mi": r_mi.sort_values("mi", ascending=False),
        "rf": r_rf.sort_values("rf_importance", ascending=False),
        "perm": r_perm.sort_values("perm_importance", ascending=False),
        "rfe": r_rfe.sort_values(["rfe_selected", "rfe_rank"], ascending=[False, True]),
        "consenso": pool,
        "rf_f1_macro_test": f1m,
    }

    del X_full, X_use, X_enc, Xtr, Xte, ytr, yte, rf, rf_perm, X_rfe
    gc.collect()

    return out


## 6) Seleccion para `tipo_violencia`
Esta celda calcula ranking multimetodo, imprime tabla Markdown Top 30 y grafica ranking final de consenso.

In [ ]:
results = {}
SAMPLE_SIZE = 50000

results["tipo_violencia"] = run_feature_selection_for_target(
    df_all=df,
    X_all=X_raw,
    target="tipo_violencia",
    sample_size=SAMPLE_SIZE,
    random_state=42,
)

top30_tv = results["tipo_violencia"]["consenso"].head(30).copy()
print("## Top 30 consenso para tipo_violencia")
print(top30_tv[["feature", "score_consenso", "score_cv", "score_mi", "score_rf", "score_perm", "score_rfe"]].to_markdown(index=False))
print(f"\nF1 macro RF baseline (muestra): {results['tipo_violencia']['rf_f1_macro_test']:.4f}")

plt.figure(figsize=(10, 8))
sns.barplot(data=top30_tv.head(20), x="score_consenso", y="feature", orient="h")
plt.title("Top 20 consenso - tipo_violencia")
plt.xlabel("Score consenso")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 7) Seleccion para `nivel_riesgo_victima`
Mismo flujo: calculo multimetodo, tabla Markdown Top 30 y grafico de ranking final.

In [ ]:
results["nivel_riesgo_victima"] = run_feature_selection_for_target(
    df_all=df,
    X_all=X_raw,
    target="nivel_riesgo_victima",
    sample_size=SAMPLE_SIZE,
    random_state=42,
)

top30_nr = results["nivel_riesgo_victima"]["consenso"].head(30).copy()
print("## Top 30 consenso para nivel_riesgo_victima")
print(top30_nr[["feature", "score_consenso", "score_cv", "score_mi", "score_rf", "score_perm", "score_rfe"]].to_markdown(index=False))
print(f"\nF1 macro RF baseline (muestra): {results['nivel_riesgo_victima']['rf_f1_macro_test']:.4f}")

plt.figure(figsize=(10, 8))
sns.barplot(data=top30_nr.head(20), x="score_consenso", y="feature", orient="h")
plt.title("Top 20 consenso - nivel_riesgo_victima")
plt.xlabel("Score consenso")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 8) Exportables CSV
Se guardan rankings por metodo y top 30 de consenso por cada target.

In [ ]:
OUT = "/home/munasqa/MAESTRIA/opencode"
for t in targets:
    results[t]["cv"].to_csv(f"{OUT}/ranking_cv_{t}.csv", index=False)
    results[t]["mi"].to_csv(f"{OUT}/ranking_mi_{t}.csv", index=False)
    results[t]["rf"].to_csv(f"{OUT}/ranking_rfimp_{t}.csv", index=False)
    results[t]["perm"].to_csv(f"{OUT}/ranking_perm_{t}.csv", index=False)
    results[t]["rfe"].to_csv(f"{OUT}/ranking_rfe_{t}.csv", index=False)
    results[t]["consenso"].head(30).to_csv(f"{OUT}/top30_consenso_{t}.csv", index=False)

print("Exportables generados en:", OUT)
